<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/09-attention-transformers.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **注意力与 Transformer** {#attention-transformers}

第 08 章比较了 recurrence、时间卷积和状态空间压缩。这些方法都通过预先确定的状态转移或局部卷积核传递信息。Attention 引入了不同的操作：一个位置通过对其他位置集合进行**内容寻址（content addressing）**来构造自己的表示。Transformer 将这种操作与投影、残差路径、归一化、前馈网络和位置信息组织起来，使整条序列可以并行训练。

本章继续使用第 08 章的 [OpenSLR SLR1 YESNO corpus](https://openslr.org/1/)。所有示例都处理本地保存的 8 kHz 录音及其八个 yes/no 标签。波形先转换为归一化 log-STFT 帧，再每四帧保留一帧，以降低二次复杂度教学示例的成本。这种时间下采样是明确声明的预处理，而不是隐藏优化。数据集只有一名男性说话者的 60 条语音，因此预测分数只用于检查机制，不代表广泛语音基准。官方 [torchaudio `YESNO` 文档](https://docs.pytorch.org/audio/main/generated/torchaudio.datasets.YESNO.html)指向同一个 OpenSLR 归档。

### **为什么引入注意力** {#why-attention-was-introduced}

早期 encoder-decoder RNN 把整条源序列压缩成一个最终隐藏向量。当源序列变长时，该向量必须保存每个解码步骤可能需要的全部细节。Attention 用一组编码器状态替代单一瓶颈，让每个输出步骤检索不同的加权组合。任意源位置到当前输出的依赖路径因此缩短，模型还可以显式给出软对齐。

Self-attention 在同一序列内部应用该原则。它用数据相关交互代替卷积的固定邻域和 recurrence 的逐步路径。对于长度 $T$ 和隐藏维度 $D$，完整 self-attention 会计算 $T\times T$ score matrix。短依赖路径与并行训练很强大，但除非施加结构或采用优化的内存日程，否则时间和内存都随 $T$ 二次增长。

Attention 本身不是完整的记忆系统，它只定义一组表示如何读取另一组表示。Transformer 还需要输入 embedding、位置或几何、非线性通道混合、残差优化路径和任务头。Attention map 的质量也依赖学习到的投影；一张热力图不会自动成为最终预测的忠实解释。

<details>
<summary><strong>PyTorch：建立共享的 YESNO attention 实验</strong></summary>

```python
import io
import math
import random
import tarfile
import wave
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

torch.set_num_threads(1)


def seed_everything(seed=909):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def read_pcm_wave(payload):
    with wave.open(io.BytesIO(payload), "rb") as stream:
        assert stream.getnchannels() == 1 and stream.getsampwidth() == 2
        sample_rate = stream.getframerate()
        samples = np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()
    return torch.tensor(samples, dtype=torch.float32) / 32768.0, sample_rate


archive_candidates = [
    Path("assets/data/waves_yesno.tar.gz"),
    Path("ipynb/Deep-Learning/assets/data/waves_yesno.tar.gz"),
]
archive = next(path for path in archive_candidates if path.exists())
window = torch.hann_window(256)
records = []
with tarfile.open(archive, "r:gz") as bundle:
    members = sorted(
        (member for member in bundle.getmembers() if member.isfile() and member.name.endswith(".wav")),
        key=lambda member: member.name,
    )
    for member in members:
        waveform, sample_rate = read_pcm_wave(bundle.extractfile(member).read())
        spectrum = torch.stft(
            waveform, n_fft=256, hop_length=160, win_length=256,
            window=window, return_complex=True,
        ).abs().transpose(0, 1)
        features = torch.log1p(spectrum)[::4]  # 20 ms hop becomes an 80 ms teaching sequence.
        labels = torch.tensor([int(value) for value in Path(member.name).stem.split("_")])
        records.append({"name": member.name, "features": features, "labels": labels})

all_idx = np.arange(len(records))
first_labels = np.array([int(record["labels"][0]) for record in records])
train_idx, holdout_idx = train_test_split(
    all_idx, test_size=0.30, random_state=808, stratify=first_labels
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=808, stratify=first_labels[holdout_idx]
)

training_frames = torch.cat([records[int(i)]["features"] for i in train_idx])
feature_mean = training_frames.mean(0)
feature_std = training_frames.std(0).clamp_min(1e-5)
for record in records:
    record["features"] = (record["features"] - feature_mean) / feature_std


class YesNoDataset(Dataset):
    def __init__(self, indices):
        self.indices = [int(i) for i in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return records[self.indices[index]]


def collate_yesno(batch):
    sequences = [item["features"] for item in batch]
    lengths = torch.tensor([len(sequence) for sequence in sequences])
    padded = pad_sequence(sequences, batch_first=True)
    valid = torch.arange(padded.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)
    labels = torch.stack([item["labels"] for item in batch])
    return padded, lengths, valid, labels


def yesno_loader(indices, shuffle=False, seed=909, batch_size=6):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        YesNoDataset(indices), batch_size=batch_size, shuffle=shuffle,
        generator=generator, collate_fn=collate_yesno,
    )


acoustic_frames, frame_lengths, frame_mask, word_labels = next(iter(yesno_loader(train_idx)))
assert len(records) == 60 and sample_rate == 8000
assert acoustic_frames.shape[0] == 6 and acoustic_frames.shape[-1] == 129
assert frame_mask.sum(1).equal(frame_lengths)
assert set(train_idx).isdisjoint(test_idx)
print({"split": (len(train_idx), len(val_idx), len(test_idx)),
       "frames": tuple(acoustic_frames.shape),
       "valid length range": (int(frame_lengths.min()), int(frame_lengths.max()))})
```

</details>

后面的每个示例都复用该划分、特征归一化和 padding mask。因此新的 attention 机制始终面对相同的序列几何，而不是每节重新生成一个恰好形状方便的随机张量。

### **Query、Key 与 Value** {#queries-keys-values}

Attention 把三个角色分开。**Query** 描述当前位置希望检索什么；**key** 描述每个候选位置可以用于匹配的地址；**value** 保存候选位置被赋予权重后返回的信息。给定 query $q$、key-value 对 $(k_i,v_i)$ 和兼容函数 $a$，attention pooling 为

$$
\alpha_i=\frac{\exp(a(q,k_i))}{\sum_j\exp(a(q,k_j))},
\qquad
\operatorname{Attention}(q,K,V)=\sum_i\alpha_i v_i.
$$

Key 与 value 属于相同候选位置，但表示不必相同。一帧声学信息可能因为某个模式适合定位，却返回另一个变换后的特征。学习型投影使这些角色依赖任务：

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
$$

逐行 softmax 使 $\alpha_i\ge0$ 且 $\sum_i\alpha_i=1$，所以每个输出是 value 向量的凸组合。投影和后续残差/FFN 仍然可以产生带符号的非线性变换；凸性只适用于一次 attention 聚合。

![Query 与 key 比较后产生权重，再利用权重汇聚对应 value。](assets/dl09-qkv.svg){fig-align="center" width="72%" fig-alt="Query、key 和 value 示意图，兼容分数转化为 value 上的 attention 权重。"}

*图片来源：[Dive into Deep Learning, Queries, Keys, and Values](https://d2l.ai/chapter_attention-mechanisms-and-transformers/queries-keys-values.html)，CC BY-SA 4.0。*

<details>
<summary><strong>PyTorch：从一条真实声学序列中检索上下文</strong></summary>

```python
seed_everything(910)
model_dimension = 32
query_projection = nn.Linear(129, model_dimension, bias=False)
key_projection = nn.Linear(129, model_dimension, bias=False)
value_projection = nn.Linear(129, model_dimension, bias=False)

sequence = acoustic_frames[0, :frame_lengths[0]]
query_frame = sequence[len(sequence) // 2:len(sequence) // 2 + 1]
query = query_projection(query_frame)                    # [1, D]
keys = key_projection(sequence)                          # [T, D]
values = value_projection(sequence)                      # [T, D]
scores = query @ keys.transpose(0, 1) / math.sqrt(model_dimension)
weights = scores.softmax(-1)
context = weights @ values

top_positions = weights[0].topk(3).indices.tolist()
assert context.shape == (1, model_dimension)
assert torch.allclose(weights.sum(-1), torch.ones(1))
assert max(top_positions) < len(sequence)
print({"query position": len(sequence) // 2,
       "top key positions": top_positions,
       "context shape": tuple(context.shape)})
```

</details>

未训练投影使选中位置没有语义；代码验证的是检索语义，而不是可解释性。任务训练后，权重会成为学习计算的一部分，但因果归因仍需要干预或梯度分析，不能把一张热力图直接当成解释。

### **缩放点积注意力** {#scaled-dot-product-attention}

对于批量矩阵 $Q\in\mathbb{R}^{T_q\times d_k}$、$K\in\mathbb{R}^{T_k\times d_k}$ 和 $V\in\mathbb{R}^{T_k\times d_v}$，scaled dot-product attention 为

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

$M$ 是加性 mask：允许位置取零，禁止位置在 softmax 前取 $-\infty$。若 query 与 key 的独立分量方差为一，它们点积的方差近似为 $d_k$。除以 $\sqrt{d_k}$ 可让 score 方差保持在一附近，避免 head dimension 增大时 softmax 越来越饱和。

数值稳定实现需要在指数运算前减去每行最大值。框架函数会在内部合并这种 log-sum-exp 变换。在低精度中，手动构造极小有限常数可能溢出或在不同 dtype 下行为不同；更适合把 boolean mask 交给维护良好的 scaled-dot-product attention 原语。

$T_qT_k$ 个 score 主导完整 attention 的内存。输出存储只有 $T_qd_v$，但朴素实现会同时物化 score 和 probability。后面的 FlashAttention 会在不改变该精确方程的情况下调整内存日程。

<details>
<summary><strong>PyTorch：在 YESNO 帧上验证缩放与稳定 softmax</strong></summary>

```python
tokens = query_projection(acoustic_frames[:2])
queries = tokens
keys_for_scale = key_projection(acoustic_frames[:2])
raw_scores = queries @ keys_for_scale.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(model_dimension)

stable_probabilities = torch.exp(scaled_scores - scaled_scores.amax(-1, keepdim=True))
stable_probabilities = stable_probabilities / stable_probabilities.sum(-1, keepdim=True)
library_probabilities = scaled_scores.softmax(-1)

raw_variance = raw_scores.var().item()
scaled_variance = scaled_scores.var().item()
assert torch.allclose(stable_probabilities, library_probabilities, atol=1e-6)
assert scaled_variance < raw_variance
assert torch.allclose(library_probabilities.sum(-1), torch.ones_like(library_probabilities[..., 0]))
print({"raw score variance": raw_variance, "scaled score variance": scaled_variance,
       "score elements": raw_scores.numel()})
```

</details>

缩放控制初始化时的 score 分布，但不能保证训练后 attention 始终良好。Query/key 范数仍可能增长，因此归一化、初始化、正则化与 attention entropy 监控仍然重要。

### **多头注意力与张量形状** {#multi-head-attention-tensor-shapes}

一张 attention map 只使用一个学习型相似度空间。Multi-head attention 创建 $H$ 个子空间，分别执行 attention，拼接结果，再投影回模型维度：

$$
\operatorname{head}_h
=\operatorname{Attention}(QW_Q^{(h)},KW_K^{(h)},VW_V^{(h)}),
$$

$$
\operatorname{MHA}(Q,K,V)
=\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O.
$$

对 $D=Hd_h$，核心形状变化是

$$
[B,T,D]
\rightarrow[B,T,H,d_h]
\rightarrow[B,H,T,d_h].
$$

Score tensor 的形状是 `[B,H,T_q,T_k]`。不同 head 不会自动学到不同语言或声学角色；对称性可能导致冗余。只有任务和优化使不同子空间有用时，head 才会分化。在固定 $D$ 下，head 数还会影响硬件效率，因为过小的 $d_h$ 可能导致矩阵运算利用率不佳。

当 query/key/value 维度相同时，主导参数量仍近似为 $4D^2$：三个输入投影加一个输出投影。在固定 $D$ 下增加 head 数改变的是分解方式，而不是这一主导参数量。

<details>
<summary><strong>PyTorch：用显式形状检查实现 multi-head attention</strong></summary>

```python
class ManualMultiHeadAttention(nn.Module):
    def __init__(self, dimension=32, heads=4):
        super().__init__()
        assert dimension % heads == 0
        self.heads = heads
        self.head_dimension = dimension // heads
        self.qkv = nn.Linear(dimension, 3 * dimension, bias=False)
        self.output = nn.Linear(dimension, dimension, bias=False)
        self.last_weights = None

    def split_heads(self, x):
        batch, time, dimension = x.shape
        return x.view(batch, time, self.heads, self.head_dimension).transpose(1, 2)

    def forward(self, x, valid_mask):
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = map(self.split_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dimension)
        scores = scores.masked_fill(~valid_mask[:, None, None, :], float("-inf"))
        weights = scores.softmax(-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(x.shape)
        self.last_weights = weights
        return self.output(context).masked_fill(~valid_mask.unsqueeze(-1), 0.0)


input_projection = nn.Linear(129, model_dimension)
acoustic_tokens = input_projection(acoustic_frames)
manual_mha = ManualMultiHeadAttention(model_dimension, heads=4)
multihead_output = manual_mha(acoustic_tokens, frame_mask)

assert multihead_output.shape == acoustic_tokens.shape
assert manual_mha.last_weights.shape == (
    acoustic_frames.shape[0], 4, acoustic_frames.shape[1], acoustic_frames.shape[1]
)
assert torch.all(multihead_output[~frame_mask] == 0)
print({"tokens": tuple(acoustic_tokens.shape),
       "heads": tuple(manual_mha.last_weights.shape),
       "head dimension": manual_mha.head_dimension})
```

</details>

在 `.view()` 前调用 `.contiguous()` 不是形式主义：transpose 会改变 stride，目标逻辑拼接可能不再位于连续内存中。Shape 和 stride 推理属于正确实现 attention 的一部分。

### **Self-Attention 与 Cross-Attention** {#self-attention-cross-attention}

在 **self-attention** 中，query、key 和 value 来自同一序列，因此每个声学帧都可以从其他帧聚合证据。如果没有位置信息或 mask，该操作是 permutation equivariant：打乱输入 token 会以相同方式打乱输出。序列顺序必须通过其他机制加入。

在 **cross-attention** 中，query 来自一个序列，key/value 来自另一个序列。Decoder token 可以查询 encoder memory，文本 token 可以查询图像 patch，小型 latent array 也可以查询大型传感器流。如果 query 长度为 $T_q$、memory 长度为 $T_k$，交互成本为 $O(T_qT_kD)$，不一定是 $O(T_k^2D)$。

这些角色是不对称的。Cross-attention 输出具有 query sequence length，因为每个 query 产生一个结果。Padding mask 通常描述无效 key/value memory 位置；无效 query 位置还需要在输出或损失中单独处理。

<details>
<summary><strong>PyTorch：让八个标签 query 读取声学 memory</strong></summary>

```python
self_attention = nn.MultiheadAttention(model_dimension, num_heads=4, batch_first=True)
self_output, self_weights = self_attention(
    acoustic_tokens, acoustic_tokens, acoustic_tokens,
    key_padding_mask=~frame_mask,
)

label_embedding = nn.Embedding(3, model_dimension)  # labels 0/1 plus start token 2
label_queries = label_embedding(word_labels)
cross_attention = nn.MultiheadAttention(model_dimension, num_heads=4, batch_first=True)
cross_output, cross_weights = cross_attention(
    label_queries, acoustic_tokens, acoustic_tokens,
    key_padding_mask=~frame_mask,
)

assert self_output.shape == acoustic_tokens.shape
assert cross_output.shape == (acoustic_frames.shape[0], 8, model_dimension)
assert cross_weights.shape == (acoustic_frames.shape[0], 8, acoustic_frames.shape[1])
assert torch.allclose(cross_weights.masked_select(~frame_mask[:, None, :]), torch.zeros_like(
    cross_weights.masked_select(~frame_mask[:, None, :])
))
print({"self": tuple(self_output.shape), "cross": tuple(cross_output.shape)})
```

</details>

这里的 label query 是 teacher-forced ground-truth embedding，只用于展示 cross-attention 接口。生成式 decoder 必须移动 target，确保某位置不会直接收到自己应该预测的标签。

### **因果、Padding 与结构化 Mask** {#causal-padding-structural-masks}

Mask 编码哪些信息通路是合法的。

- **Padding mask** 阻止真实 query 读取人工 padding key。其形状通常为 `[B,T_k]`，并跨 head 与 query position 广播。
- **Causal mask** 阻止位置 $t$ 读取 $j>t$ 的 key。它是下三角结构，用于保持自回归分解。
- **Structural mask** 只允许局部窗口、block、图边、模态关系或应用定义的连接。

对于加性 mask $M_{ij}$，

$$
M_{ij}=\begin{cases}
0,&\text{允许交互},\\
-\infty,&\text{禁止交互}.
\end{cases}
$$

Mask 必须在 softmax **之前**应用。事后把 probability 乘零会让剩余行和小于一，除非再次归一化，而且被屏蔽位置可能已经影响数值最大值。如果一个 query row 全部被 mask，对全 $-\infty$ 做 softmax 没有定义；应移除该 query、提供合法 sentinel 通路，或显式清零。

训练 target 也必须正确 shift。即使有 causal mask，如果用于预测 $y_t$ 的输入位置直接包含 $y_t$，仍会发生泄漏。Decoder input 应是 start token 后接 $y_{<t}$。

<details>
<summary><strong>PyTorch：验证 padding 与 causal 信息边界</strong></summary>

```python
padding_weights = manual_mha.last_weights
masked_padding_weights = padding_weights.masked_select(~frame_mask[:, None, None, :])
assert torch.all(masked_padding_weights == 0)

start = torch.full((word_labels.shape[0], 1), 2, dtype=torch.long)
decoder_inputs = torch.cat([start, word_labels[:, :-1]], dim=1)
decoder_tokens = label_embedding(decoder_inputs)
causal_mask = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1)
causal_attention = nn.MultiheadAttention(model_dimension, 4, batch_first=True)
causal_output, causal_weights = causal_attention(
    decoder_tokens, decoder_tokens, decoder_tokens,
    attn_mask=causal_mask,
)

future_entries = causal_weights[:, causal_mask]
assert causal_output.shape == decoder_tokens.shape
assert torch.all(future_entries == 0)
assert torch.equal(decoder_inputs[:, 1:], word_labels[:, :-1])
print({"padding weights checked": masked_padding_weights.numel(),
       "future weights checked": future_entries.numel()})
```

</details>

可靠测试应在固定合法输入时修改禁止输入，并验证较早输出不变。只查看 mask tensor 无法发现 target shift、broadcast 或 cache position 错误。

### **位置编码与 RoPE** {#positional-encoding-rope}

没有位置信息时，self-attention 把 token 当作集合。绝对 sinusoidal encoding 加入确定性向量：

$$
PE_{p,2i}=\sin\left(p/10000^{2i/D}\right),
\qquad
PE_{p,2i+1}=\cos\left(p/10000^{2i/D}\right).
$$

不同频率使线性组合可以表示相对偏移，而且不需要学习位置表。学习型 absolute embedding 可以适应训练长度，但超过范围时需要外推策略。

Relative-position 方法会根据距离改变 attention logit。ALiBi 加入 head-specific 线性距离偏置。**Rotary Position Embedding（RoPE）**则按位置角度旋转每一对 query/key 特征。对于一组二维特征，

$$
R(m\theta)^\top R(n\theta)=R((n-m)\theta),
$$

所以位置 $m$ 的 query 与位置 $n$ 的 key 之间点积依赖相对偏移 $n-m$，同时每个向量仍获得绝对旋转。不同特征对使用不同频率。

![RoPE 旋转 query/key 特征对，使点积显式包含相对位置。](assets/dl09-rope.svg){fig-align="center" width="76%" fig-alt="Rotary position embedding 示意图，展示位置相关的 query/key 旋转与相对角度点积。"}

*图片来源：依据 [Su et al., RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864) 绘制的本地教学图。*

RoPE 不会自动保证可靠的上下文外推。频率谱、最大训练长度、缩放方法、数值精度和任务都会影响结果。Position interpolation 与 frequency rescaling 会改变行为，必须在超出训练长度的输入上评估，而不能直接假定有效。

<details>
<summary><strong>PyTorch：应用 sinusoidal encoding 并验证 RoPE 相对平移</strong></summary>

```python
def sinusoidal_encoding(length, dimension):
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    frequencies = torch.exp(
        torch.arange(0, dimension, 2, dtype=torch.float32) * (-math.log(10000.0) / dimension)
    )
    encoding = torch.zeros(length, dimension)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding


def apply_rope(x, positions):
    dimension = x.shape[-1]
    frequencies = 1.0 / (10000.0 ** (torch.arange(0, dimension, 2) / dimension))
    angles = positions.float().unsqueeze(-1) * frequencies
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated_even = even * angles.cos() - odd * angles.sin()
    rotated_odd = even * angles.sin() + odd * angles.cos()
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)


ordered_tokens = acoustic_tokens[0, :frame_lengths[0]]
absolute_tokens = ordered_tokens + sinusoidal_encoding(len(ordered_tokens), model_dimension)
assert not torch.allclose(absolute_tokens[0], absolute_tokens[1])

q = ordered_tokens[3:4]
k = ordered_tokens[11:12]
score_a = (apply_rope(q, torch.tensor([3])) * apply_rope(k, torch.tensor([11]))).sum()
score_b = (apply_rope(q, torch.tensor([8])) * apply_rope(k, torch.tensor([16]))).sum()
assert torch.allclose(score_a, score_b, atol=1e-5)
print({"sequence length": len(ordered_tokens), "relative offset": 8,
       "shift-invariant RoPE score": score_a.item()})
```

</details>

两个位置都平移五，因此相对偏移仍为八，上述等式成立。在 multi-head attention 中，RoPE 通常应用于 query 与 key 投影，而不是 value。

### **Transformer Encoder 与 Decoder Block** {#transformer-encoder-decoder-blocks}

原始 Transformer 是 encoder-decoder 架构。每个 encoder block 包含 self-attention 和 position-wise FFN；每个 decoder block 还增加 masked self-attention、读取 encoder memory 的 cross-attention，以及 FFN。残差连接与归一化围绕这些子层。

对于 encoder 输入 $X\in\mathbb{R}^{B\times T_s\times D}$ 和 decoder state $Y\in\mathbb{R}^{B\times T_t\times D}$：

- encoder self-attention 使用 $Q=K=V=X$，输出长度为 $T_s$；
- decoder causal self-attention 使用 $Q=K=V=Y$，输出长度为 $T_t$；
- cross-attention 使用 $Q=Y$、$K=V=X_{enc}$，输出长度为 $T_t$。

![Transformer 把堆叠 encoder self-attention block 与因果 masked decoder、encoder-decoder attention 组合起来。](assets/dl09-transformer.svg){fig-align="center" width="72%" fig-alt="包含 self-attention、cross-attention、前馈网络、残差路径和位置编码的 Transformer encoder-decoder 架构。"}

*图片来源：[Dive into Deep Learning, The Transformer Architecture](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html)，CC BY-SA 4.0。*

训练时，decoder 接收 shift 后的 target。在输出步骤 $u$，模型看到 start token 和标签 $y_{<u}$，然后预测 $y_u$。Teacher forcing 使所有目标位置能够在 causal mask 下并行训练，尽管推理仍然逐 token 生成。

<details>
<summary><strong>PyTorch：构建声学 encoder 与标签 decoder</strong></summary>

```python
class TinySpeechTransformer(nn.Module):
    def __init__(self, input_size=129, dimension=32, heads=4):
        super().__init__()
        self.dimension = dimension
        self.input_projection = nn.Linear(input_size, dimension)
        self.label_embedding = nn.Embedding(3, dimension)
        encoder_layer = nn.TransformerEncoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.0,
            batch_first=True, norm_first=True,
        )
        decoder_layer = nn.TransformerDecoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.0,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.output = nn.Linear(dimension, 2)

    def forward(self, source, source_mask, target_input):
        source_tokens = self.input_projection(source)
        source_tokens = source_tokens + sinusoidal_encoding(source.shape[1], self.dimension)
        memory = self.encoder(source_tokens, src_key_padding_mask=~source_mask)
        target_tokens = self.label_embedding(target_input)
        target_tokens = target_tokens + sinusoidal_encoding(target_input.shape[1], self.dimension)
        causal = torch.triu(torch.ones(target_input.shape[1], target_input.shape[1], dtype=torch.bool), 1)
        decoded = self.decoder(
            target_tokens, memory, tgt_mask=causal,
            memory_key_padding_mask=~source_mask,
        )
        return self.output(decoded), memory


seed_everything(911)
speech_transformer = TinySpeechTransformer()
teacher_input = torch.cat([
    torch.full((word_labels.shape[0], 1), 2, dtype=torch.long),
    word_labels[:, :-1],
], dim=1)
label_logits, encoder_memory = speech_transformer(acoustic_frames, frame_mask, teacher_input)
sequence_loss = F.cross_entropy(label_logits.flatten(0, 1), word_labels.flatten())
sequence_loss.backward()

assert label_logits.shape == (acoustic_frames.shape[0], 8, 2)
assert encoder_memory.shape == acoustic_tokens.shape
assert all(parameter.grad is not None for parameter in speech_transformer.parameters())
print({"memory": tuple(encoder_memory.shape), "label logits": tuple(label_logits.shape),
       "teacher-forced loss": sequence_loss.item()})
```

</details>

该示例验证完整信息路径，但没有训练语音识别器。真正实验需要在训练划分上优化，并在未见说话者上评估 sequence error，而不能从一次 backward pass 推断质量。

### **残差路径、归一化与前馈网络** {#residual-normalization-feed-forward}

Attention 混合的是**位置之间**的信息。FFN 使用共享权重，在每个位置独立变换通道：

$$
\operatorname{FFN}(x)=W_2\,\phi(W_1x+b_1)+b_2.
$$

内部维度通常是 $D$ 的若干倍，使 FFN 占据大量参数和 FLOPs。GLU、GEGLU 和 SwiGLU 等门控变体会让一个分支乘上学习门；它们改变通道计算，却不改变序列混合。

残差连接保留恒等通路，并要求每个子层返回维度 $D$。Layer normalization 按 token 稳定特征尺度。两种常见排列是

$$
\text{post-norm: }x_{l+1}=\operatorname{LN}(x_l+F(x_l)),
$$

$$
\text{pre-norm: }x_{l+1}=x_l+F(\operatorname{LN}(x_l)).
$$

Post-norm 与原始 Transformer 一致，但深层优化可能困难，因为每条恒等路径都经过归一化。Pre-norm 通常改善梯度流，是深层模型的常见选择，但也会改变激活增长和最终归一化要求。残差缩放、谨慎初始化和 normalization 变体仍是重要设计选择。

<details>
<summary><strong>PyTorch：在音频 token 上比较 pre-norm 与 post-norm 梯度路径</strong></summary>

```python
class TransformerSubBlock(nn.Module):
    def __init__(self, dimension=32, heads=4, pre_norm=True):
        super().__init__()
        self.pre_norm = pre_norm
        self.norm1 = nn.LayerNorm(dimension)
        self.norm2 = nn.LayerNorm(dimension)
        self.attention = nn.MultiheadAttention(dimension, heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dimension, 64), nn.GELU(), nn.Linear(64, dimension))

    def forward(self, x, valid):
        if self.pre_norm:
            normalized = self.norm1(x)
            attended = self.attention(normalized, normalized, normalized,
                                      key_padding_mask=~valid, need_weights=False)[0]
            x = x + attended
            return x + self.ffn(self.norm2(x))
        attended = self.attention(x, x, x, key_padding_mask=~valid, need_weights=False)[0]
        x = self.norm1(x + attended)
        return self.norm2(x + self.ffn(x))


gradient_records = {}
for name, pre_norm in (("pre", True), ("post", False)):
    seed_everything(912)
    block_input = acoustic_tokens[:2].detach().clone().requires_grad_(True)
    block = TransformerSubBlock(pre_norm=pre_norm)
    output = block(block_input, frame_mask[:2])
    loss = output[frame_mask[:2]].pow(2).mean()
    loss.backward()
    gradient_records[name] = block_input.grad.norm().item()
    assert torch.isfinite(block_input.grad).all()

print({"input gradient norms": gradient_records,
       "FFN parameters": sum(p.numel() for p in TransformerSubBlock().ffn.parameters())})
```

</details>

一个随机初始化 block 无法建立普遍的梯度排名。该实验验证两种计算路径，并提供深度扫描所需的日志模式。关于可训练性的结论需要多层网络、受控初始化和重复运行。

### **Encoder-Only、Decoder-Only 与 Encoder-Decoder 模型** {#encoder-decoder-model-families}

Transformer 家族的主要差异在于信息边界和训练目标。

**Encoder-only** 模型对已观测输入使用双向 self-attention，适合整个输入都可用时的分类、检索 embedding 和 token-level prediction。Masked-token pretraining 会隐藏部分输入，但剩余上下文仍是双向的。

**Decoder-only** 模型使用 causal self-attention，并优化自回归似然

$$
p(x_{1:T})=\prod_{t=1}^{T}p(x_t\mid x_{<t}).
$$

同一接口支持生成与 in-context conditioning，但每个预测 token 都必须遵守因果 mask。Prompt token 是已观察上下文，生成 token 会继续扩展上下文。

**Encoder-decoder** 模型让自回归 target 以独立编码的 source 为条件：

$$
p(y_{1:U}\mid x_{1:T})=\prod_{u=1}^{U}p(y_u\mid y_{<u},x_{1:T}).
$$

当输入输出的模态、长度或角色不同时，例如 speech-to-text 与翻译，这种结构很自然。Encoder 可以双向处理 source，decoder 仍保持 causal。

<details>
<summary><strong>PyTorch：在 YESNO 上训练 encoder-only 首词分类器</strong></summary>

```python
class SpeechEncoderClassifier(nn.Module):
    def __init__(self, input_size=129, dimension=32, heads=4):
        super().__init__()
        self.projection = nn.Linear(input_size, dimension)
        layer = nn.TransformerEncoderLayer(
            dimension, heads, dim_feedforward=64, dropout=0.1,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(dimension, 2)

    def forward(self, x, valid):
        tokens = self.projection(x) + sinusoidal_encoding(x.shape[1], 32)
        encoded = self.encoder(tokens, src_key_padding_mask=~valid)
        pooled = (encoded * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True)
        return self.head(pooled)


@torch.no_grad()
def classifier_accuracy(model, indices):
    model.eval()
    correct = total = 0
    for xb, _, valid, labels in yesno_loader(indices):
        prediction = model(xb, valid).argmax(1)
        correct += int((prediction == labels[:, 0]).sum())
        total += len(labels)
    return correct / total


seed_everything(913)
encoder_classifier = SpeechEncoderClassifier()
optimizer = torch.optim.AdamW(encoder_classifier.parameters(), lr=2e-3, weight_decay=1e-3)
for epoch in range(8):
    encoder_classifier.train()
    for xb, _, valid, labels in yesno_loader(train_idx, shuffle=True, seed=913):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(encoder_classifier(xb, valid), labels[:, 0])
        loss.backward()
        nn.utils.clip_grad_norm_(encoder_classifier.parameters(), 1.0)
        optimizer.step()

validation_accuracy = classifier_accuracy(encoder_classifier, val_idx)
assert 0.0 <= validation_accuracy <= 1.0
print({"encoder-only validation accuracy": validation_accuracy,
       "parameters": sum(p.numel() for p in encoder_classifier.parameters())})
```

</details>

极小验证集使分数非常离散。该实验的目的，是证明同一批归一化音频、mask 和 split 可以支持端到端 Transformer 训练。模型家族应根据合法上下文和输出结构选择，而不是根据流行度选择。

### **KV Cache 与自回归推理** {#kv-cache-autoregressive-inference}

在 teacher-forced training 中，causal decoder 并行处理所有目标位置。自回归推理时，它生成一个 token、追加到序列并重复。朴素实现会在每一步重新计算整个 prefix 的 key/value 投影。

**KV cache** 保存每一 decoder layer 已产生的 key 和 value tensor。在步骤 $t+1$，只有新 token 被投影为 $q_{t+1},k_{t+1},v_{t+1}$；query 读取缓存的 $K_{1:t+1},V_{1:t+1}$。Cache 消除了历史 token 的重复投影和重复处理，但不会消除新 query 与所有保留 key 的点积。

![KV cache 保存过去的 key/value 投影，每个新 token 只追加一组 K/V 并发出一个 query。](assets/dl09-kv-cache.svg){fig-align="center" width="78%" fig-alt="自回归解码示意图，过去的 key-value 投影保存在持久 cache 中，一个新 query 读取它们。"}

*图片来源：依据增量 causal-attention 计算绘制的本地教学图。*

对于 $L$ 层、batch size $B$、缓存长度 $T$、key/value head 数 $H_{kv}$、head dimension $d_h$ 和每元素 $s$ bytes，cache memory 近似为

$$
M_{KV}=2LBTH_{kv}d_hs.
$$

因子二对应 key 和 value。Multi-query attention 在所有 query head 间共享一个 K/V head；grouped-query attention 使用中间数量，从而降低 cache memory 和带宽。Beam search 需要谨慎复制或共享 cache state，continuous batching 则要追踪每条序列的位置与淘汰。

<details>
<summary><strong>PyTorch：证明 cached decoding 与完整 causal attention 一致</strong></summary>

```python
seed_everything(914)
label_sequence = word_labels[0]
label_tokens = label_embedding(label_sequence.unsqueeze(0))[0]
q_layer = nn.Linear(model_dimension, model_dimension, bias=False)
k_layer = nn.Linear(model_dimension, model_dimension, bias=False)
v_layer = nn.Linear(model_dimension, model_dimension, bias=False)
q_all, k_all, v_all = q_layer(label_tokens), k_layer(label_tokens), v_layer(label_tokens)

full_scores = q_all @ k_all.T / math.sqrt(model_dimension)
full_scores = full_scores.masked_fill(torch.triu(torch.ones(8, 8, dtype=torch.bool), 1), float("-inf"))
full_output = full_scores.softmax(-1) @ v_all

cached_keys, cached_values, cached_outputs = [], [], []
for step in range(len(label_tokens)):
    cached_keys.append(k_all[step:step + 1])
    cached_values.append(v_all[step:step + 1])
    key_cache = torch.cat(cached_keys, dim=0)
    value_cache = torch.cat(cached_values, dim=0)
    step_scores = q_all[step:step + 1] @ key_cache.T / math.sqrt(model_dimension)
    cached_outputs.append(step_scores.softmax(-1) @ value_cache)
cached_output = torch.cat(cached_outputs, dim=0)

assert torch.allclose(full_output, cached_output, atol=1e-6)
cache_elements = 2 * len(label_tokens) * model_dimension
print({"tokens": len(label_tokens), "cached K/V elements": cache_elements,
       "maximum output error": (full_output - cached_output).abs().max().item()})
```

</details>

真实 cache 还会为每层保存一组 tensor，并对新 query/key 应用正确位置编码。Position ID 偏移、causal mask 不一致、旧 cache 条目和 beam 重排是常见推理错误。

### **FlashAttention 与高效 Attention 原则** {#flashattention-efficient-attention}

朴素 attention 把完整 score matrix $S=QK^\top$ 和 probability matrix $P=\operatorname{softmax}(S)$ 写入高带宽内存（HBM），之后再读取并计算 $PV$。在现代加速器上，这些内存传输可能比算术本身更昂贵。

FlashAttention 是一种**精确且 IO-aware** 的算法。它把 query、key、value 分块装入片上 SRAM，一次计算一个 score block，并维护在线 softmax 统计量。对于跨 key block 处理的一行，保存 running maximum $m$、denominator $\ell$ 和未归一化 output numerator $o$。新 block 最大值为 $m_b$ 时，

$$
m'=\max(m,m_b),
$$

$$
\ell'=e^{m-m'}\ell+\sum_j e^{s_j-m'},
\qquad
o'=e^{m-m'}o+\sum_j e^{s_j-m'}v_j.
$$

所有 block 完成后输出 $o/\ell$。使用新最大值重缩放，可以在不让完整行同时驻留内存的情况下保持精确稳定 softmax。Backward pass 可以重新计算选定中间量，而不保存完整 probability matrix。

![FlashAttention 把 Q/K/V 分块送入片上内存，更新精确 online-softmax 统计量，不在 HBM 中物化完整 score matrix。](assets/dl09-flashattention-tiling.svg){fig-align="center" width="78%" fig-alt="FlashAttention 内存图，包含 HBM block、SRAM score tile、online softmax state 与精确输出。"}

*图片来源：依据 [Dao et al., FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135) 绘制的本地教学图。*

FlashAttention 改变 kernel schedule，而不是 dense all-pairs interaction 的数学复杂度。实际收益取决于序列长度、head dimension、dtype、硬件、mask、dropout 和可用内核。PyTorch 的 `scaled_dot_product_attention` 可以调度到优化 backend；生产代码应优先使用它，并通过 profiling 确认实际选中的 backend。

<details>
<summary><strong>PyTorch：用分块 online softmax 复现精确 attention</strong></summary>

```python
def tiled_online_attention(q, k, v, block_size=16):
    outputs = []
    scale = 1.0 / math.sqrt(q.shape[-1])
    for q_start in range(0, len(q), block_size):
        q_block = q[q_start:q_start + block_size]
        running_max = torch.full((len(q_block), 1), float("-inf"))
        running_sum = torch.zeros(len(q_block), 1)
        running_output = torch.zeros(len(q_block), v.shape[-1])
        for k_start in range(0, len(k), block_size):
            k_block = k[k_start:k_start + block_size]
            v_block = v[k_start:k_start + block_size]
            score_block = q_block @ k_block.T * scale
            new_max = torch.maximum(running_max, score_block.amax(-1, keepdim=True))
            old_scale = torch.exp(running_max - new_max)
            probabilities = torch.exp(score_block - new_max)
            running_output = old_scale * running_output + probabilities @ v_block
            running_sum = old_scale * running_sum + probabilities.sum(-1, keepdim=True)
            running_max = new_max
        outputs.append(running_output / running_sum)
    return torch.cat(outputs, dim=0)


attention_input = acoustic_tokens[0, :min(int(frame_lengths[0]), 48)]
q_flash = query_projection(acoustic_frames[0, :len(attention_input)])
k_flash = key_projection(acoustic_frames[0, :len(attention_input)])
v_flash = value_projection(acoustic_frames[0, :len(attention_input)])
reference = (q_flash @ k_flash.T / math.sqrt(model_dimension)).softmax(-1) @ v_flash
tiled = tiled_online_attention(q_flash, k_flash, v_flash, block_size=16)

assert torch.allclose(reference, tiled, atol=1e-5)
assert 16 * 16 < len(q_flash) * len(k_flash) or len(q_flash) <= 16
print({"sequence length": len(q_flash),
       "full score elements": len(q_flash) * len(k_flash),
       "largest teaching tile": min(16, len(q_flash)) * min(16, len(k_flash)),
       "maximum error": (reference - tiled).abs().max().item()})
```

</details>

这个循环展示 online-softmax 代数，但比普通 PyTorch 更慢，因为 Python 启动了许多小操作。FlashAttention 的价值来自融合硬件实现和精心设计的 IO 日程。

### **文本之外的 Transformer** {#transformers-beyond-text}

Attention 需要的是 token，而不一定是单词。Tokenization 决定 Transformer 将要建模的几何。

- **Vision：**图像变为固定 patch、分层窗口、region proposal 或学习型 visual token；二维位置和多尺度结构很重要。
- **Audio：**波形采样、spectrogram frame 或时频 patch 变为 token；长录音通常需要下采样、局部前端或流式 chunk。
- **Time series：**每个时间步可以包含多个变量，也可以把变量本身作为 token；缺失值、不规则时间戳和因果预测边界需要显式编码。
- **Multimodal system：**模态专用 encoder 产生 token，再通过 cross-attention、共享 self-attention 或小型 latent bottleneck 交互。

相同的 attention 方程不意味着模态可以互换。文本 token 是离散且语义学习的；图像 patch 有二维邻域；声学帧表示重叠窗口；传感器事件可能具有不规则时间间隔。Position、augmentation、masking objective 与 output head 必须反映这些差异。

对音频而言，减少 token 数常常至关重要。把连续 $P$ 帧分组，会把长度从 $T$ 降到约 $T/P$，并把完整 attention score 数量减少约 $P^2$，但也会损失时间分辨率。卷积下采样可以学习这种压缩；固定拼接则让权衡更直观。

<details>
<summary><strong>PyTorch：把 YESNO spectrogram frame 转换为声学 patch</strong></summary>

```python
patch_size = 4
single_audio = acoustic_frames[0, :frame_lengths[0]]
usable_length = (len(single_audio) // patch_size) * patch_size
frame_patches = single_audio[:usable_length].reshape(-1, patch_size * 129)
patch_projection = nn.Linear(patch_size * 129, model_dimension)
audio_patch_tokens = patch_projection(frame_patches)

original_score_elements = usable_length ** 2
patched_score_elements = len(audio_patch_tokens) ** 2
assert audio_patch_tokens.shape == (usable_length // patch_size, model_dimension)
assert original_score_elements // patched_score_elements == patch_size ** 2
print({"original frames": usable_length, "patch tokens": len(audio_patch_tokens),
       "attention score reduction": original_score_elements / patched_score_elements})
```

</details>

Patch size 为四时，score 数量下降 $16\times$，但模型在 patch projection 前无法关注单个 frame。高效 tokenization 和高效 attention 解决成本的不同部分，应该联合评估。

### **本章对比与总结** {#chapter-comparison-summary}

Attention 是内容寻址的聚合算子。只有把它与位置结构、mask、逐位置非线性通道计算、残差优化路径和任务特定信息边界结合，Transformer 才成为完整架构。

| 组件或家族 | 主要作用 | 张量/成本重点 | 典型失败 |
|---|---|---|---|
| query、key、value | 分离检索请求、地址和内容 | $Q:[B,T_q,D]$，$K,V:[B,T_k,D]$ | 把 attention weight 当作因果解释 |
| scaled dot product | 稳定的内容相似度与汇聚 | score `[B,H,T_q,T_k]` | 缺少 $\sqrt{d_k}$ 缩放或 mask 不稳定 |
| multi-head attention | 多个学习型交互子空间 | `[B,T,D]` reshape 为 `[B,H,T,d_h]` | 隐蔽 transpose/view 错误或 head 冗余 |
| self-attention | 同一序列内部交互 | $O(T^2D)$ dense interaction | 没有 position 时不含顺序 |
| cross-attention | 一个序列读取另一个序列 | $O(T_qT_kD)$ | 混淆 query/memory 长度与 mask |
| padding/causal mask | 约束合法信息路径 | softmax 前广播 | target leakage 或全 mask row |
| sinusoidal/RoPE | 注入顺序与相对偏移 | 位置-频率特征对 | 未测试长序列就假定可外推 |
| encoder block | 双向 source 表示 | self-attention 加 FFN | padding 污染 |
| decoder block | 因果生成，可选 source 条件 | causal self-attention 加可选 cross-attention | target 未 shift 或 cache position 错误 |
| pre-/post-norm | 控制残差优化路径 | 逐 token 特征统计 | 把单 block 行为外推到深层网络 |
| encoder-only | 全上下文表示 | 一条已观察序列 | 不适用于严格因果部署 |
| decoder-only | 自回归建模 | 增长的上下文和 cache | 二次上下文交互与误差累积 |
| encoder-decoder | 条件序列生成 | 独立 source/target 长度 | 跨模态对齐或解码瓶颈 |
| KV cache | 复用过去 key/value 投影 | $2LBTH_{kv}d_hs$ bytes | memory 增长、过期 cache、beam 重排 |
| FlashAttention | 减少 dense attention HBM 流量 | 分块精确 online softmax | 把 IO 节省误认为次二次数学 |

YESNO 实验在本章形成一条连续路线：

1. 把官方波形转为固定且只用训练集归一化的声学序列，并建立显式 padding mask。
2. 用一帧真实输入作为 query，从其他 frame 中读取信息，解释 Q/K/V 语义。
3. 缩放点积、拆分 head，并验证每个张量形状。
4. 让标签 query cross-attend 到声学 memory，同时保持 source padding 边界。
5. 加入 causal 与位置结构，再组装完整 teacher-forced encoder-decoder。
6. 在同一 split 上训练 encoder-only 分类器，把机制连接到端到端目标。
7. 证明增量 KV cached decoding 与完整 causal attention 一致。
8. 证明分块 online softmax 与朴素 dense attention 完全一致。
9. 通过显式 patching 减少音频序列长度，并计算损失的时间分辨率。

实践选择规则不是“所有序列都使用 Transformer”。只有当直接内容相关交互值得其内存和计算成本时才使用 attention；根据合法上下文和输出分解选择 encoder-only、decoder-only 或 encoder-decoder；同时测量 tokenization、score matrix 大小、cache memory、kernel backend 和部署延迟。第 10 章将继续讨论图神经网络，其中允许的交互由关系结构而不是稠密序列决定。